In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import os
from pathlib import Path

notebook_path = "/u/skarmakar1/version_check/llm_steering-main/sk"
sys.path.append(notebook_path)

In [3]:
import torch
import numpy as np

from inversion_utils import *
import pickle
from sklearn.model_selection import train_test_split

In [4]:
SEED = 0
# SEED = 1

torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
np.random.seed(SEED)

torch.backends.cudnn.benchmark = True 
torch.backends.cuda.matmul.allow_tf32 = True

LLM = namedtuple('LLM', ['language_model', 'tokenizer', 'processor', 'name', 'model_type'])

In [5]:
model_type = 'llama'
# model_type = 'qwen'

# MODEL_VERSION = '3'
MODEL_VERSION = '3.1'
# MODEL_VERSION = '3.3'

MODEL_SIZE = '8B'
# MODEL_SIZE = '70B'

llm = select_llm(model_type, MODEL_VERSION=MODEL_VERSION, MODEL_SIZE=MODEL_SIZE)

Loading meta-llama/Meta-Llama-3.1-8B-Instruct


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [6]:
hidden_layers = list(range(-1, -llm.language_model.config.num_hidden_layers, -1))
print(hidden_layers)

[-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]


Testing

In [7]:
# coef = 0.65
# coef = 0.7
coef = 0.75
# coef = 0.8
# coef = 0.9
max_tokens = 200

prompts = ["It seems the family next doors has fallen ill."] # ('sympathetic', 'unsympathetic', 'social')
# prompts = ["I am walking home alone at night, I am a little scared.",] # ('brave', 'cowardly', 'social')
# prompts = ["You are designing a new vehicle. Describe its size and weight."] # ('wide', 'narrow', 'physical')
# prompts = ["Explain the triple point of water."] # ('deep', 'superficial', 'complexity')
# prompts = ["Analyze the following assertion. Assertion: You can always take the ducks from the public park, they are free."] # ('legal', 'illegal', 'logic') - works better after inversion
# prompts = ["Is the following statement correct or not?"] # ('correct', 'incorrect', 'logic') - weird!!!
# prompts = ["Describe this sound."] # ('agonizing', 'soothing', 'texture')
# prompts = ["Describe the following event or entity. Subject: ."] # ('long-term', 'short-term', 'time')
# prompts = ["I am walking home alone at night, is it safe?"] # ('safe', 'dangerous', 'state') - almost

# prompts = ["What is weight of a football?"] # ('precise', 'inaccurate', 'complexity')

c1 = "sympathetic"
c2 = "unsympathetic"

category = "social"

c1_controller = load_controller(llm, c1, path=f'../all_gitignore/directions_adjectives_llama/{category}/')
# c1_controller = load_controller(llm, c1, path=f'../all_gitignore/directions_adjectives_llama_class0/{category}/')
orig_c1 = c1_controller.directions
c2_controller = load_controller(llm, c2, path=f'../all_gitignore/directions_adjectives_llama/{category}/')
# c2_controller = load_controller(llm, c2, path=f'../all_gitignore/directions_adjectives_llama_class0/{category}/')
orig_c2 = c2_controller.directions



# c1_controller = load_controller(llm, c1, path=f'../all_gitignore/directions_adjectives_llama/{category}/')
# orig_c1 = c1_controller.directions
# c2_controller = load_controller(llm, c2, path=f'../all_gitignore/directions_adjectives_llama/{category}/')
# orig_c2 = c2_controller.directions

print(f"Coef: {coef}")

out = test_concept_vector(c1_controller, concept=c1, prompts=prompts, coef=coef, max_tokens=max_tokens)
out = test_concept_vector(c2_controller, concept=c2, prompts=prompts, coef=coef, max_tokens=max_tokens, orig=False)


# coef = 0.8

# inv_c1_lrr = apply_auto(c1_controller.directions, lrr_models)
# c1_controller.directions = inv_c1_lrr
# out = test_concept_vector(c1_controller, concept=f"combined inverted {c1}, class {category}, original, coef {coef}", prompts=prompts, coef=coef, max_tokens=max_tokens, orig=False)

# inv_c2_lrr = apply_auto(c2_controller.directions, lrr_models)
# c2_controller.directions = inv_c2_lrr
# out = test_concept_vector(c2_controller, concept=f"combined inverted {c2}, class {category}, original, coef {coef}", prompts=prompts, coef=coef, max_tokens=max_tokens, orig=False)

Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 16
M_batch_size         : 2048
n_components         : 1

Detector found
Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 16
M_batch_size         : 2048
n_components         : 1

Detector found
Coef: 0.75

========================== No Control ==========================
It seems the family next doors has fallen ill.
-----------------------------------------------------
I'm so sorry to hear that the family next door has fallen ill.  That can be a really stressful and concerning situation for everyone involved. 

H

In [8]:
# coef = 0.65
# coef = 0.7
coef = 0.75
# coef = 0.8
# coef = 0.9
max_tokens = 200

prompts = ["It seems the family next doors has fallen ill."] # ('sympathetic', 'unsympathetic', 'social')
# prompts = ["I am walking home alone at night, I am a little scared.",] # ('brave', 'cowardly', 'social')
# prompts = ["You are designing a new vehicle. Describe its size and weight."] # ('wide', 'narrow', 'physical')
# prompts = ["Explain the triple point of water."] # ('deep', 'superficial', 'complexity')
# prompts = ["Analyze the following assertion. Assertion: You can always take the ducks from the public park, they are free."] # ('legal', 'illegal', 'logic') - works better after inversion
# prompts = ["Is the following statement correct or not?"] # ('correct', 'incorrect', 'logic') - weird!!!
# prompts = ["Describe this sound."] # ('agonizing', 'soothing', 'texture')
# prompts = ["Describe the following event or entity. Subject: ."] # ('long-term', 'short-term', 'time')
# prompts = ["I am walking home alone at night, is it safe?"] # ('safe', 'dangerous', 'state') - almost

# prompts = ["What is weight of a football?"] # ('precise', 'inaccurate', 'complexity')

c1 = "sympathetic"
c2 = "unsympathetic"

category = "social"

# c1_controller = load_controller(llm, c1, path=f'../all_gitignore/directions_adjectives_llama/{category}/')
c1_controller = load_controller(llm, c1, path=f'../all_gitignore/directions_adjectives_llama_class0/{category}/')
orig_c1 = c1_controller.directions
# c2_controller = load_controller(llm, c2, path=f'../all_gitignore/directions_adjectives_llama/{category}/')
c2_controller = load_controller(llm, c2, path=f'../all_gitignore/directions_adjectives_llama_class0/{category}/')
orig_c2 = c2_controller.directions



# c1_controller = load_controller(llm, c1, path=f'../all_gitignore/directions_adjectives_llama/{category}/')
# orig_c1 = c1_controller.directions
# c2_controller = load_controller(llm, c2, path=f'../all_gitignore/directions_adjectives_llama/{category}/')
# orig_c2 = c2_controller.directions

print(f"Coef: {coef}")

out = test_concept_vector(c1_controller, concept=c1, prompts=prompts, coef=coef, max_tokens=max_tokens)
out = test_concept_vector(c2_controller, concept=c2, prompts=prompts, coef=coef, max_tokens=max_tokens, orig=False)


# coef = 0.8

# inv_c1_lrr = apply_auto(c1_controller.directions, lrr_models)
# c1_controller.directions = inv_c1_lrr
# out = test_concept_vector(c1_controller, concept=f"combined inverted {c1}, class {category}, original, coef {coef}", prompts=prompts, coef=coef, max_tokens=max_tokens, orig=False)

# inv_c2_lrr = apply_auto(c2_controller.directions, lrr_models)
# c2_controller.directions = inv_c2_lrr
# out = test_concept_vector(c2_controller, concept=f"combined inverted {c2}, class {category}, original, coef {coef}", prompts=prompts, coef=coef, max_tokens=max_tokens, orig=False)

Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 16
M_batch_size         : 2048
n_components         : 1

Detector found
Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 16
M_batch_size         : 2048
n_components         : 1

Detector found
Coef: 0.75

========================== No Control ==========================
It seems the family next doors has fallen ill.
-----------------------------------------------------
I'm so sorry to hear that the family next door has fallen ill.  That can be a really stressful and concerning situation for everyone involved. 

H

In [ ]:
# elif dataset_label == 'languages':
#     dataset = utils.language_dataset(llm, 'english', concept)
# elif dataset_label == 'supervised_languages':
#     dataset = utils.supervised_language_dataset('data/languages', [concept], llm.tokenizer)